# AI Video Studio - Wan 2.2 Video Generation on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Farmington1/comfy/blob/main/colab/ComfyUI_Video_Generation.ipynb)

Generate videos with **Wan 2.2** (text-to-video and image-to-video) using **ComfyUI**, driven entirely from this notebook.

**How this stays within Google Colab's rules:** the free Colab tier prohibits interacting primarily through a *web UI* (ngrok/cloudflared tunnels to the ComfyUI interface). This notebook instead runs ComfyUI **headless on localhost** and submits workflows through its **local API from these cells** - a compliant, notebook-driven path. (On a *paid* plan you may optionally open the real ComfyUI web UI; see the docs.)

**What you need:** a Google account and a GPU runtime (`Runtime -> Change runtime type -> GPU`). An A100 (40/80 GB) is ideal; T4/L4 work with the smaller model + fast preview.

Run the cells top to bottom.

## Step 1 - Connect a GPU and load the project

First set your repository URL (the public **github.com** mirror of this repo - Colab's loader cannot read private/enterprise Git hosts), then run the cell. It clones the repo, installs helper dependencies and checks your GPU.

In [ ]:
import os, sys, subprocess

# >>> EDIT THIS to your public github.com repo (used only for cloning in Colab) <<<
REPO_URL = os.environ.get('AVS_REPO_URL', 'https://github.com/Farmington1/comfy.git')
REPO_BRANCH = os.environ.get('AVS_REPO_BRANCH', 'main')
PROJECT_DIR = '/content/ai-video-studio'

def sh(cmd):
    print('$', ' '.join(cmd)); subprocess.run(cmd, check=True)

# Clone the repo if we are not already inside it.
if not os.path.exists('scripts/setup.py'):
    if not os.path.exists(PROJECT_DIR):
        sh(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, PROJECT_DIR])
    os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())

# Install the lightweight orchestration dependencies.
sh([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

# Quick GPU banner (full validation runs in the next cell).
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'])

### Validate the environment

Detects your GPU model, VRAM, CUDA/PyTorch, system RAM and free disk, then recommends a model tier and the ComfyUI VRAM flag. It warns loudly if the runtime is too small.

In [ ]:
!python scripts/validate_environment.py

## Step 2 - (Recommended) Mount Google Drive for persistence

Model weights, generated videos, configs and uploaded images are saved under `MyDrive/AI-Video-Studio/` so they survive runtime resets. Set `USE_DRIVE = False` to run ephemerally.

In [ ]:
USE_DRIVE = True  # set False to skip Google Drive
import os
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.environ['AVS_USE_DRIVE'] = '1'
        print('Google Drive mounted.')
    except Exception as e:
        print('Drive mount skipped/unavailable:', e)
        os.environ['AVS_USE_DRIVE'] = '0'
else:
    os.environ['AVS_USE_DRIVE'] = '0'

## Step 3 - Install ComfyUI and download the Wan 2.2 models

Pick a tier (or leave `auto` to use the recommendation from Step 1):

| Tier | Models | Powers | Rough size |
|------|--------|--------|-----------|
| `base` | Wan 2.2 **TI2V 5B** + encoder + VAE | `wan_text_to_video`, `wan_image_to_video`, `fast_preview` | ~17 GB |
| `quality` | Wan 2.2 **14B T2V** (high+low noise) + encoder + VAE | `high_quality` | ~35 GB |
| `gguf` | Quantized 14B T2V (low VRAM) | low-VRAM GPUs | ~20 GB |

This step is **idempotent** - re-running skips anything already installed/downloaded, and resumes partial downloads. It also starts the ComfyUI server (headless, localhost).

In [ ]:
TIER = 'auto'  # 'auto' | 'base' | 'quality' | 'gguf' | 'i2v_14b'

cmd = ['python', 'scripts/setup.py', '--tier', TIER]
if os.environ.get('AVS_USE_DRIVE') != '1':
    cmd.append('--no-drive')
print('Running:', ' '.join(cmd))
import subprocess; subprocess.run(cmd, check=True)

## Step 4 - Confirm ComfyUI is ready

In [ ]:
!python scripts/launch_comfyui.py status

## Step 5 - Generate your first video (text-to-video)

Edit the prompt and settings, then run. The job is submitted to the local ComfyUI API and the finished MP4 is saved to `ComfyUI/output` and copied to Google Drive (`MyDrive/AI-Video-Studio/outputs`).

- **Fastest first result:** set `WORKFLOW = 'fast_preview'`.
- **Best quality (needs the `quality` tier + a big GPU):** `WORKFLOW = 'high_quality'`.

In [ ]:
import sys, importlib
sys.path.insert(0, 'scripts')
import generate; importlib.reload(generate)

WORKFLOW = 'wan_text_to_video'   # 'fast_preview' | 'wan_text_to_video' | 'high_quality'

params = {
    'positive_prompt': 'A cinematic aerial shot flying over a misty mountain range at sunrise, golden light breaking through the clouds, slow smooth camera motion.',
    'seed': 42,
    # Optional overrides (otherwise the workflow defaults are used):
    # 'width': 1280, 'height': 704, 'length': 121, 'steps': 20, 'cfg': 5.0, 'fps': 24,
}

result = generate.run_generation(WORKFLOW, params)
print('Local outputs:', [str(p) for p in result['local']])
print('Drive copies :', [str(p) for p in result['drive']])

### Preview / download the result

In [ ]:
from IPython.display import HTML, display
import base64, pathlib

def show_video(path):
    data = pathlib.Path(path).read_bytes()
    b64 = base64.b64encode(data).decode()
    display(HTML(f'<video width=512 controls autoplay loop src="data:video/mp4;base64,{b64}"></video>'))

if result['local']:
    show_video(result['local'][0])
    # To download to your laptop:
    try:
        from google.colab import files; files.download(str(result['local'][0]))
    except Exception as e:
        print('Download helper unavailable:', e)
else:
    print('No output produced - check the ComfyUI log in the logs folder.')

## Step 6 - Image-to-video (optional)

Upload a reference image, then animate it with a motion prompt using the Wan 2.2 5B model.

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose one image
image_path = next(iter(uploaded)) if uploaded else None
print('Uploaded:', image_path)

In [ ]:
import importlib, generate; importlib.reload(generate)

i2v_params = {
    'positive_prompt': 'The subject comes to life with gentle, natural motion; slow cinematic camera push-in, soft realistic lighting.',
    'input_image': image_path,
    'seed': 123,
}
i2v_result = generate.run_generation('wan_image_to_video', i2v_params)
if i2v_result['local']:
    show_video(i2v_result['local'][0])

## Step 7 - Shut down to save compute units

When you are done, stop ComfyUI and **disconnect the runtime** so Colab stops consuming compute units.

1. Run the cell below to stop the ComfyUI server.
2. Then: `Runtime -> Disconnect and delete runtime` (or uncomment the `runtime.unassign()` line).

Your models and videos remain safe in Google Drive.

In [ ]:
!python scripts/launch_comfyui.py stop
# Fully release the GPU (uncomment to auto-disconnect):
# from google.colab import runtime; runtime.unassign()